In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

!pip install facenet-pytorch

src_path = "/content/drive/MyDrive/11/src"

if src_path not in sys.path:
    sys.path.append(src_path)

import cv2
import numpy as np
import time
from biometric_system import IntegratedBiometricSystem, Config
from base64 import b64decode
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow


In [5]:

# Funzione Javascript per Colab (dal tuo codice originale)
def capture_image_colab(quality=0.8):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Cattura Foto';
        div.appendChild(capture);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
      ''')
    display(js)
    try:
        data = eval_js('takePhoto({})'.format(quality))
        if not data: return None
        binary = b64decode(data.split(',')[1])
        return cv2.imdecode(np.frombuffer(binary, np.uint8), -1)
    except Exception: return None

# --- ISTANZIA ED ESEGUI IL SISTEMA ---
cfg = Config()
# 1. Istanzia il sistema
bio_system = IntegratedBiometricSystem(cfg)

# 2. Esegui la pipeline
# Potrai scegliere Enrollment, Identification, Verification da input testuale
bio_system.run_pipeline(capture_func=capture_image_colab)

>>> Inizializzazione Sistema Biometrico...
[LIB] Detector inizializzato su cuda. Controllo Yaw + Pitch attivo.
Inizializzazione Modello su cuda...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.
ATTENZIONE: Nessuna Gallery trovata. Necessario Enrollment.
ERRORE: Pesi Liveness non trovati in ./content/drive/MyDrive/11/11/Consegna parziale/models/best_weights_spoof_gr_11

--- SELEZIONA MODALITÀ ---
1. Enrollment (Registra nuovo utente)
2. Identification (Chi sono? - Open Set)
3. Verification (Io sono l'utente X, verificami)


KeyboardInterrupt: Interrupted by user